### Group Members:

- Name, matriculation number
- Name, matriculation number
- Name, matriculation number

# Assignment 3 - Convolutional Network and Transfer Learning

The goal of this exercise is to learn how to train a small convolutional neural network (CNN) and fine-tune this trained network in transfer learning tasks.

Our CNN has the following layers:

1. 2D convolutional layer with $Q_1$ channels, kernel size $7\times7$, stride 1 and padding 0.
2. 2D maximum pooling with pooling size $3\times3$ and stride 3
3. `Tanh` activation
4. 2D convolutional layer with $Q_2$ channels, kernel size $5\times5$, stride 1 and padding 2.
5. 2D maximum pooling with pooling size $2\times2$ and stride 2
6. `Tanh` activation
7. A flattening layer to turn the 3D image into a 1D vector
8. A fully-connected layer with the appropriate number of inputs and $O$ outputs.

For this exercise, we will switch to an implementation in `PyTorch`.
We will get used to some concepts in `PyTorch`, such as relying on the `torch.tensor` data structure, implementing the network, the loss functions, the training loop, and accuracy computation, which we will apply to categorical classification.
We will see how backpropagation and weight update will be done automatically by `torch`.

Please make sure that all your variables are compatible with `torch`.
For example, you cannot mix `torch.tensor`s and `numpy.array`s in any part of the code.

The CNN will be trained on the `letters` from EMNIST datasets and fine-tuned with the `digits` from the same dataset.

## Theoretical Sections

In this section, we analyze the key properties of a CNN focusing on its spatial characteristics and parameterization. Specifically, we address the following fundamental aspects:  

1. Receptive Field Computation  
2. Feature Map Dimensions
3. Learnable Parameter Count  
4. Derivatives of Pooling

Through these computations, we gain insights into the network's ability to capture spatial hierarchies and its overall complexity.

Besides, we also point out some possible problems that implementing the training process may face.

#### Task 1.1: Compute Receptive Field

Compute the receptive field size of an element of the final pooling layers before the flattening operation.

Hints:

- Consider one location in the last feature map, i.e., after the second pooling layer. Go backwards through the layers and compute the size of the receptive field for each operation. 
- Look at the graphics from the slides of Lecture 5 to understand how to compute receptive fields for convolution and pooling layers.

Answer: 

To compute the receptive field, we work backwards from one element in the final pooling layer using the formula:

**r_input = (r_output - 1) × stride + kernel_size**

**Working backwards through the layers:**

1. **Start:** One element (1×1) in the output of the second pooling layer

2. **Through 2nd Max Pooling (kernel 2×2, stride 2):**
   - r = (1 - 1) × 2 + 2 = 2×2

3. **Through 2nd Convolution (kernel 5×5, stride 1, padding 2):**
   - r = (2 - 1) × 1 + 5 = 6×6

4. **Through 1st Max Pooling (kernel 3×3, stride 3):**
   - r = (6 - 1) × 3 + 3 = 18×18

5. **Through 1st Convolution (kernel 7×7, stride 1, padding 0):**
   - r = (18 - 1) × 1 + 7 = 24×24

**Final Answer: The receptive field size is 24×24 pixels in the original input image.**

#### Task 1.2: Compute Learnable Parameters

Given that the input image has shape $28\times28$,

1. Compute feature map size, i.e., the number of inputs for the last fully connected layer.
2. Estimate roughly how many learnable parameters this network has by analytically computing and adding the number of parameters in each layer. Express the estimation in terms of $Q_1, Q_2, O$.

Answer:

**Part 1: Feature map size (number of inputs to the FC layer)**

Using the output size formula: $\text{out} = \lfloor \frac{\text{in} - k + 2p}{s} \rfloor + 1$

| Layer | Computation | Output Shape |
|-------|-------------|-------------|
| Input | — | $1 \times 28 \times 28$ |
| Conv1 ($7 \times 7$, $s{=}1$, $p{=}0$) | $\lfloor(28{-}7)/1\rfloor{+}1 = 22$ | $Q_1 \times 22 \times 22$ |
| MaxPool1 ($3 \times 3$, $s{=}3$) | $\lfloor(22{-}3)/3\rfloor{+}1 = 7$ | $Q_1 \times 7 \times 7$ |
| Conv2 ($5 \times 5$, $s{=}1$, $p{=}2$) | $\lfloor(7{-}5{+}4)/1\rfloor{+}1 = 7$ | $Q_2 \times 7 \times 7$ |
| MaxPool2 ($2 \times 2$, $s{=}2$) | $\lfloor(7{-}2)/2\rfloor{+}1 = 3$ | $Q_2 \times 3 \times 3$ |
| Flatten | $Q_2 \times 3 \times 3$ | $9Q_2$ |

The number of inputs to the fully connected layer is $9Q_2$.

**Part 2: Learnable parameters**

- **Conv1**: weights $Q_1 \times 1 \times 7 \times 7 = 49Q_1$, biases $Q_1$ → total $50Q_1$
- **Conv2**: weights $Q_2 \times Q_1 \times 5 \times 5 = 25Q_1Q_2$, biases $Q_2$ → total $25Q_1Q_2 + Q_2$
- **FC**: weights $O \times 9Q_2 = 9Q_2O$, biases $O$ → total $9Q_2O + O$
- MaxPool and Tanh layers have **no** learnable parameters.

$$\text{Total} = 50Q_1 + 25Q_1Q_2 + Q_2 + 9Q_2O + O$$


#### Task 1.3: Derivatives of Pooling

Given two pooling methods:

1. Average pooling: $$a_k=\frac1R \sum\limits_{j=1}^R \hat a_{{R(k-1)+j}}$$
2. Maximum pooling: $$a_k=\max\limits_{j \in \{1,\ldots, R\}} \hat a_{{R(k-1)+j}}$$

Compute $$\frac{\partial a_k}{\partial \hat a_{R(k-1)+j}}$$

Answer:

**1. Average Pooling:**

Each input appears linearly with coefficient $\frac{1}{R}$, so the derivative is constant:

$$\frac{\partial a_k}{\partial \hat{a}_{R(k-1)+j}} = \frac{1}{R} \quad \forall \, j \in \{1, \ldots, R\}$$

The gradient is distributed **uniformly** across all elements in the pooling window.

**2. Maximum Pooling:**

The output equals the largest element. The derivative is non-zero only for the maximum element:

$$\frac{\partial a_k}{\partial \hat{a}_{R(k-1)+j}} = \begin{cases} 1 & \text{if } j = \arg\max_{j' \in \{1,\ldots,R\}} \hat{a}_{R(k-1)+j'} \\ 0 & \text{otherwise} \end{cases}$$

The gradient is **sparse** — only the maximum element receives the full gradient, all others receive zero.


#### Task 1.4

For a randomly initialized network, what is the estimated probabilities for the classes in binary classification and categorical classification tasks?
To which expected loss value would this turn to?

Make use of the loss functions provided below:

$$
\mathcal{J}^{\text{BCE}} = - \frac{1}{N} \sum_{n=1}^{N} \left[ t^{[n]} \log(y^{[n]}) + (1 - t^{[n]}) \log(1 - y^{[n]}) \right]
$$

$$
\mathcal{J}^{\text{CCE}} = - \frac{1}{N} \sum_{n=1}^{N} \sum_{o=1}^{O} t_o^{[n]} \log y_o^{[n]}
$$


Answer:

**Estimated probabilities for a randomly initialized network:**

- **Binary classification:** The network output (after sigmoid) is approximately random, so $y^{[n]} \approx \frac{1}{2}$.
- **Categorical classification ($O$ classes):** The network output (after softmax) assigns roughly equal probability to each class, so $y_o^{[n]} \approx \frac{1}{O}$.

**Expected loss values:**

**1. Binary Cross-Entropy (BCE):**

Assuming balanced classes, approximately half the samples have $t^{[n]}=1$ and half $t^{[n]}=0$. With $y^{[n]} = \frac{1}{2}$:

$$\mathcal{J}^{\text{BCE}} = -\left[\frac{1}{2}\log\frac{1}{2} + \frac{1}{2}\log\frac{1}{2}\right] = -\log\frac{1}{2} = \ln 2 \approx 0.693$$

**2. Categorical Cross-Entropy (CCE):**

With one-hot targets, only the true class $o^*$ contributes to the inner sum ($t_{o^*}^{[n]}=1$, all others 0). With $y_{o^*}^{[n]} = \frac{1}{O}$:

$$\mathcal{J}^{\text{CCE}} = -\log\frac{1}{O} = \ln O$$


#### Task 1.5:

Given the example loss and accuracy plots, compared with the standard plots, in which the learning rate is 0.001 and the computation for loss and accuracy is correct, analyze the possible problems for the other plots.

Answer:

**Standard Plots (Baseline):**
- Loss decreases smoothly from ~1.7 to ~0.3
- Accuracy increases smoothly from ~0.72 to ~0.92
- Train and validation curves track closely with minimal gap
- This represents correct implementation with appropriate learning rate

**Example 1 - Unstable Training:**

**Problem: Learning rate too high or numerical instability**

**Observations:**
- Extremely noisy/erratic loss curves with  oscillations (0.75-1.0 range)
- Validation loss shows large spikes and discontinuous jumps
- Accuracy plateaus around 0.87 but with significant noise
- Training is unstable and not converging smoothly

**Likely causes:**
- **Learning rate too high** ($\eta \gg 0.001$): The optimizer overshoots minima, causing oscillations
- **Gradient explosion**: Unstable gradients leading to erratic updates
- **Batch size too small**: High variance in gradient estimates

**Example 2 - Learning Rate Too Low:**

**Problem: Learning rate too low**

**Observations:**
- Loss starts at ~3.25 (close to $\ln 26 \approx 3.258$ for random initialization)
- Loss decreases extremely slowly, only reaching ~2.3 after 20 epochs
- Accuracy increases very slowly, barely reaching 0.5 after 20 epochs
- The network is learning, but progress is slow

**Likely causes:**
- **Learning rate too small** ($\eta \ll 0.001$): Weight updates are too conservative
- Network needs many more epochs to converge (possibly 100+ epochs)
- Could also indicate optimizer momentum is too low or absent

**Example 3 - Incorrect Loss Computation:**

**Problem: Incorrect loss calculation/accumulation**

**Observations:**
- Loss values are unrealistically small (0.004-0.025 range)
- The shape/pattern looks similar to standard plots but scaled down by ~100×
- Accuracy curve appears normal (~0.92)
- Initial loss ~0.027 is far too low (should be ~3.258 for random init)

**Likely causes:**
- **Missing `.item()` when accumulating loss**: Storing tensors instead of scalar values
- **Incorrect normalization**: Dividing by wrong count (e.g., total samples instead of batches, or vice versa)
- **Loss not averaged per sample**: Using reduction='sum' instead of reduction='mean' in loss function
- **Accumulation error**: Not properly summing losses across batches before averaging

## Coding

**<font color='red' size='5'>This section has to be submitted by 11:59 p.m. on Wednesday, April 22, to be graded.</font>**

Before we start, we should ensure that we have activated CUDA -- otherwise training might take very long.
In Google Colaboratory:

1. Check the options Runtime -> Change Runtime Type on top of the page.
2. In the popup window, select hardware accelerator GPU.

Afterward, the following command should run successfully:

In [ ]:
import torch
import torchvision

if torch.cuda.is_available():
    print("Successfully enabled CUDA processing")
    device = torch.device("cuda")
else:
    print("CUDA processing not available. Things will be slow :-(")
    device = torch.device("cpu")

### Dataset

In PyTorch, a dataset stores a list of input and target tensors $(X^{[n]}, T^{[n]})$.
We will make use of EMNIST dataset for this assignment.
In particular, we will use the letters to train the CNN for classification task and then fine-tune this CNN using the digits.

In the **EMNIST** dataset, the inputs are $X^{[n]} \in \mathbb R^{28\times28}$. 
In case of the `letters` split, the labels are $T^{[n]} \in \{1,\ldots,26\}$.
For `digits` split, the labels $T^{[n]} \in \{0,\ldots,9\}$ correspond to the digit.

More precisely, the data in the dataset is provided in the form of `PIL.Image.Image`, which represents an image class with some more functionality, and pixel values in range $[0, 255]$.
To convert these images into `torch.Tensor`'s in range $[0,1]$, we can use the `torchvision.transforms.ToTensor` transform.
Furthermore, in `PyTorch` batches are created from datasets using the `torch.utils.data.DataLoader` class.

#### Task 2.1: Dataset Loading

Here, we use the letters (`split="letters"`) in EMNIST dataset of gray images for categorical classification, and digits (`split="digits"`) for transfer learning.

Write a function that returns the training and the validation set of the dataset, using the given `transform` and `split`.

Note:

Targets for `letters` range $[1,26]$ by default, which will cause problem when using the loss desired function (which accepts $[0,25]$ instead) in `PyTorch`.
Set `target_transform` to a function that can shift the target by subtracting 1 when `split="letters".`

In [ ]:
def datasets(split,transform):
    """Instantiates and returns the training and validation sets for the EMNIST dataset for the given split.

    Args:
    split: "letters" or "digits" to obtain the EMNIST letters or the EMNIST MNIST dataset
    transform: The transform to apply to all input images

    Returns:
        trainset: The training partition of the requested dataset
        validset: The validation (aka the test split) of the requested dataset
    """

    if split == "letters":
        # apply lambda function to change the range of targets from [1,26] to [0,25]
        target_transform = lambda x: x - 1
    else:
        target_transform = None

    trainset = torchvision.datasets.EMNIST(root='./data', split=split, train=True, download=True, transform=transform, target_transform=target_transform)
    validset = torchvision.datasets.EMNIST(root='./data', split=split, train=False, download=True, transform=transform, target_transform=target_transform)

    return trainset, validset

#### Test 1: Data Types

Create the dataset with `transform=None`.
Check that all inputs are of type `PIL.Image.Image`, and all targets are integral.

In [ ]:
import PIL
splits_ = ["letters","digits"]
for split_ in splits_:
    ts_, vs_ = datasets(split=split_,transform=None)

    for x_,t_ in ts_:
        # check datatype of input x
        assert isinstance(x_, PIL.Image.Image)
        # check datatype of target t
        assert isinstance(t_, int)

#### Task 2.2: Data Visulization
Create the dataset with `transform=None`.
Use `matplotlib` to make a plot with 4 rows and 10 columns.

Since all images in EMNIST dataset have been encoded with mixed-up horizontal and vertical axes, we want to plot the original image, as well as the version with a fixed orientation.

Specifically, in the first row plot 10 images of letter trainset, and in the second row, plot them again with correct orientation. In the third row plot 10 images of digit trainset, and in the fourth row, plot them again with correct orientation.

In [ ]:
# sometimes the kernel crashes when using matplotlib and pytorch together, the following solves this
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import matplotlib.pyplot
import numpy
matplotlib.pyplot.rcParams['image.cmap'] = 'gray'


# Load dataset without transformation (to show original images)
train_letters, _ = datasets(split="letters",transform=None)
train_digits, _ = datasets(split="digits",transform=None)

# Function to fix exchanged horizontal and vertical axes
def fix_orientation(img):
    return numpy.array(img).T

# Visualization
fig, axes = matplotlib.pyplot.subplots(4, 10, figsize=(15, 6))

# Select 10 samples
for j in range(10):
    # Load letter
    img_letter, _ = train_letters[j]
    fixed_letter = fix_orientation(img_letter)

    # Load digit
    img_digit, _ = train_digits[j]
    fixed_digit = fix_orientation(img_digit)

    # Plot raw and fixed letters
    axes[0, j].imshow(img_letter)
    axes[1, j].imshow(fixed_letter)

    # Plot raw and fixed digits
    axes[2, j].imshow(img_digit)
    axes[3, j].imshow(fixed_digit)

    # Remove axes to show only images
    for i in range(4):
        axes[i, j].axis("off")

#### Task 2.3: Data Loaders

Create two datasets with two splits each, using a simple `ToTensor` input image transform.
For each dataset, create two data loaders, one for the training set and one for the validation set.
The training batch size should be $B=64$, for the validation set, you can choose any batch size of your choice.

In [ ]:
# select appropriate transform
transform = torchvision.transforms.ToTensor()

# create datasets
trainset_letters, validset_letters = datasets(split="letters",transform=transform)
trainset_digits, validset_digits = datasets(split="digits",transform=transform)

# instantiate data loaders
trainloader_letters = torch.utils.data.DataLoader(trainset_letters, batch_size=64, shuffle=True)
validloader_letters = torch.utils.data.DataLoader(validset_letters, batch_size=64, shuffle=False)

trainloader_digits = torch.utils.data.DataLoader(trainset_digits, batch_size=64, shuffle=True)
validloader_digits = torch.utils.data.DataLoader(validset_digits, batch_size=64, shuffle=False)

#### Test 2: Batches

Check that all inputs and targets are of type `torch.Tensor`.

Check that all input values are in range $[0,1]$.

Check that all target values are in range $[0,25]$ for letters and $[0,9]$ for digits.

In [ ]:
for x_,t_ in trainloader_letters:
    # check datatype, size and content of x
    assert isinstance(x_, torch.Tensor)
    assert torch.all(x_ >= 0) and torch.all(x_ <= 1)
    # check datatype, size and content of t
    assert isinstance(t_, torch.Tensor)
    assert torch.all(t_ >= 0) and torch.all(t_ <= 25)


for x_,t_ in trainloader_digits:
    # check datatype, size and content of x
    assert isinstance(x_, torch.Tensor)
    assert torch.all(x_ >= 0) and torch.all(x_ <= 1)

    # check datatype, size and content of t
    assert isinstance(t_, torch.Tensor)
    assert torch.all(t_ >= 0) and torch.all(t_ <= 9)

### Convolutional Network Training

For training and evaluating the network, we will rely on standard functionality in PyTorch.
We will use the standard categorical cross-entropy loss together with a stochastic gradient descent optimizer.
For training, we will use the batched implementation of the dataset, for which we perform one update step for each training batch.

For each epoch, we will compute and save the average loss and accuracy for the full training set and validation set.
This will be used to visualize the training process of CNN.

#### Task 2.4: Training and Validation Loop


Implement a function that takes the network, train loader, validation loader, the number of epochs, the learning rate, and the momentum.
Select the correct loss function for categorical classification and SGD optimizer.
Iterate the following steps for the given number of epochs:

1. Train the network with all batches of the training data.
2. Compute the train set loss, train set accuracy, validation set loss, validation set accuracy.
3. Store all in lists, for later visualization of CNN training process.

Finally, return the lists of train losses and accuracies, as well as validation losses and accuracies.

Notes:

- Make sure that you train on the training data only, and **not** on the validation data.

- When you iterate over validation set, please use `with torch.no_grad():` and loop on validloader inside it. This disables gradient computation and therefore saves memory.

- While saving loss values, please use `.item()`.

- Make sure that you divide the summed loss and accuracy values by the correct count.

In [ ]:
def training_loop(network, trainloader, validloader, epochs, lr, momentum):
    """Trains the given network for the given number of epochs on the given training data using the given learning rate and momentum.
    After each epoch, the validation loss and accuracy are computed using the provided validation data

    Args:
    network: A convolutional network wuth the correct number of output nodes
    trainloader: The data loader of the training set
    validloader: The data loader for the vaildation set
    eopchs: The number of epochs to train for
    lr: The learing rate to apply
    momentum: the momentum parameter for the learning

    Returns:
        train_loss_list: The average training losses computed during the epochs
        train_acc_list: The average training accuracies computed during the epochs
        val_loss_list: The average validation losses computed after the epochs
        val_acc_list: The average validation accuracies computed after the epochs
    """

    # move the network to the selected device
    network.to(device)

    # select loss function and optimizer
    loss_function = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(network.parameters(), lr=lr, momentum=momentum)

    # collect loss values and accuracies over the training epochs
    train_loss_list, train_acc_list = [], []
    val_loss_list, val_acc_list = [], []

    for epoch in range(epochs):
        # train network on training data
        train_loss, train_correct, train_total = 0, 0, 0
        for x, t in trainloader:
            # put data to device
            device_x, device_t = x.to(device), t.to(device)
            # train
            optimizer.zero_grad()
            outputs = network(device_x)
            loss = loss_function(outputs, device_t)
            loss.backward()
            optimizer.step()
            # calculate training accuracies and losses for current batch
            train_loss += loss.item() * device_x.size(0)
            _, predicted = torch.max(outputs, 1)
            train_correct += (predicted == device_t).sum().item()
            train_total += device_t.size(0)
        # append training accuracies and losses for current epoch
        train_loss_list.append(train_loss / train_total)
        train_acc_list.append(train_correct / train_total)

        # validate network on validation data
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for x, t in validloader:
                # put data to device
                device_x, device_t = x.to(device), t.to(device)
                # compute validation loss
                outputs = network(device_x)
                loss = loss_function(outputs, device_t)
                val_loss += loss.item() * device_x.size(0)
                # compute validation accuracy
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == device_t).sum().item()
                val_total += device_t.size(0)

        # append validation accuracies and losses for current epoch
        val_loss_list.append(val_loss / val_total)
        val_acc_list.append(val_correct / val_total)

        # print training loss, accuracy, validation loss, accuracy for current epoch
        print(f"Epoch {epoch+1}/{epochs}, "
              f"Train Loss: {train_loss_list[-1]:.4f}, Train Acc: {train_acc_list[-1]:.4f}, "
              f"Val Loss: {val_loss_list[-1]:.4f}, Val Acc: {val_acc_list[-1]:.4f}")

    # return the collected lists
    return train_loss_list, train_acc_list, val_loss_list, val_acc_list

#### Task 2.5: Convolutional Network

We will rely on `torch.nn.Sequential` to create networks with particular lists of consecutive layers.

Implement a function that generates a convolutional network with the following layers:

1. 2D convolutional layer with $Q_1$ channels, kernel size $7\times7$, stride 1 and padding 0.
2. 2D maximum pooling with pooling size $3\times3$ and stride 3
3. `tanh` activation
4. 2D convolutional layer with $Q_2$ channels, kernel size $5\times5$, stride 1 and padding 2.
5. 2D maximum pooling with pooling size $2\times2$ and stride 2
6. `tanh` activation
7. A flattening layer to turn the 3D feature map into a 1D vector
8. A fully-connected layer with the appropriate number of inputs and $O$ outputs.

In [ ]:
def convolutional(Q1, Q2, O):
    """Instantiates a convolutional network used for categorical classification.

    Args:
    Q1: The number of output channels of the first convolutional layer
    Q2: The number of output channels of the second convolutional layer
    O: The number of output logits of the final fully-connected layer

    Returns:
        An instance of a convvolutional network with the given parameters
    """
    return torch.nn.Sequential(
        torch.nn.Conv2d(in_channels=1, out_channels=Q1, kernel_size=7, stride=1, padding=0),
        torch.nn.MaxPool2d(kernel_size=3, stride=3),
        torch.nn.Tanh(),
        torch.nn.Conv2d(in_channels=Q1, out_channels=Q2, kernel_size=5, stride=1, padding=2),
        torch.nn.MaxPool2d(kernel_size=2, stride=2),
        torch.nn.Tanh(),
        torch.nn.Flatten(),
        torch.nn.Linear(Q2 * 3 * 3, O)
    )

#### Test 3: Network Implementation

Create a network with an arbitrary shape.

Create a batch that follows input dimensions.

Forward the batch through the network.

Check that the output dimensions fit.

In [ ]:
net_ = convolutional(3,4,6)
batch_ = torch.rand((8,1,28,28))
output_ = net_(batch_)
assert output_.shape == (8,6)

#### Task 2.6: Convolutional Training

Create a convolutional network with $Q_1=16$ and $Q_2=32$ convolutional channels and $O=26$ output neurons.
Train the network for 5 epochs with $\eta=0.01$, $\text{momentum}=0.9$ and store the obtained train losses, accuracies, test losses and accuracies.

If you want, you can also train for 20 epochs, the training time will increase accordingly -- it might take up to 30 minutes on the CPU.

In [ ]:
# instantiate initial network according to the above description
initial_network = convolutional(Q1=16, Q2=32, O=26)
# train the network on the Letters training data
train_loss, train_acc, val_loss, val_acc = training_loop(initial_network, trainloader_letters, validloader_letters, epochs=10, lr=0.01, momentum=0.9)

#### Task 2.7: Plotting

Plot the loss values in one plot and accuracy values into another plot.

In [ ]:
matplotlib.pyplot.figure(figsize=(10,3))
ax = matplotlib.pyplot.subplot(121)
# plot training and validation loss values of network over epochs
ax.plot(train_loss, label='Train Loss')
ax.plot(val_loss, label='Validation Loss')
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.set_title('Loss over Epochs')
ax.legend()


ax =  matplotlib.pyplot.subplot(122)
# plot training and validation accuracy values of network over epochs
ax.plot(train_acc, label='Train Accuracy')
ax.plot(val_acc, label='Validation Accuracy')
ax.set_xlabel('Epochs')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy over Epochs')
ax.legend()


### Transfer Learning

We will make use of above trained CNN, which can be used to classify 26 handwritten characters, and fine-tune this CNN as to the digits classification task instead.

#### Task 2.8: Pre-trained Network Instantiation

Make a copy of the trained CNN.

Freeze the feature layers of the copied network.

Notes:

- To freeze layers, you can simply disable gradient computation for all learnable parameters of the network via `parameter.requires_grad = False`.

In [ ]:
import copy
# copy trained convolutional network
network_copy = copy.deepcopy(initial_network).to("cpu")

# Freeze only the feature layers (not the final fully-connected layer)
for i, layer in enumerate(network_copy):
    if i < len(network_copy) - 1:  # All layers except the last one
        for param in layer.parameters():
            param.requires_grad = False

#### Task 2.9: Network Implementation

We want to modify the network such that we extract the logits for the 10 classes from the last fully-connected layer of the network.

Implement a function that replaces the current last linear layer of the pre-trained network with a new linear layer that has $O$ units ($O$ represents the number of classes in our dataset).

In [ ]:
def replace_last_layer(network, O=10):
    """Replaces the final layer with the given number of output units, and reinitializes the weights

    Args:
    network: A convolutional network, where the last layer is a fully-connected layer
    O: The number of output neurons for the new fully-connected layer

    Returns:
        network: The input network with the last layer replaced
    """

    # replace the last linear layer with a new layer
    in_features = network[-1].in_features
    network[-1] = torch.nn.Linear(in_features, O)

    return network

#### Test 4: Last Layer Dimensions

This test ensures that the function return a network having the correct number of input and output units in the last layer.

In [ ]:
O_ = 6

test_network_ = replace_last_layer(network_copy, O=O_)
assert test_network_[-1].out_features == O_
assert test_network_[-1].in_features == 288

#### Task 2.10: Network Fine-Tuning with Frozen Layers

Create a network that has feature layers frozen with $10$ output units.
Fine-tune the created network for 2 epochs on our digits data (`trainloader_digits, validloader_digits`) using the previous function, and a smaller learning rate of $\eta=0.001$.

In [ ]:
# replace network layer
network_with_frozen_layers = replace_last_layer(network_copy, O=10)
# fine-tune last layer with the digits data
_ = training_loop(network_with_frozen_layers, trainloader_digits, validloader_digits, epochs=2, lr=0.001, momentum=0.9)

#### Test 5: Frozen Layers

Check that all layers of the fine-tuned network that contain weights and biases (except for the last fully-connected layer) have not been modified by the training, and that the final layer differs in shape.

In [ ]:
for i in range(len(initial_network)-1):
    if hasattr(initial_network[i], "weight"):
        assert torch.allclose(initial_network[i].weight, network_with_frozen_layers[i].weight)
    if hasattr(initial_network[i], "bias"):
        assert torch.allclose(initial_network[i].bias, network_with_frozen_layers[i].bias)

assert initial_network[-1].weight.shape != network_with_frozen_layers[-1].weight.shape
assert initial_network[-1].bias.shape != network_with_frozen_layers[-1].bias.shape

#### Task 2.11: Test Set Predictions

Go over the validation set of the Digits dataset once again and extract predictions of the network.

In [ ]:
predictions = []
targets = []
with torch.no_grad():
    for x, t in validloader_digits:
        # put data onto device
        device_x, device_t = x.to(device), t.to(device)
        # extract logits from the network
        logits = network_with_frozen_layers(device_x)
        # obtain predicted class
        predicted_class = logits.argmax(dim=1)

        # store predicted class
        predictions.extend(predicted_class.cpu().tolist())

        # store target class
        targets.extend(device_t.cpu().tolist())


#### Task 2.12: Confusion Matrix Plotting

Finally, we want to plot the confusion matrix of the validation set.
For this, we can make use of the `sklearn.metrics.confusion_matrix` to compute the confusion matrix.
You can utilize `sklearn.metrics.ConfusionMatrixDisplay` for displaying the confusion matrix, or `pyplot.imshow` and adding the according labels.

Note:

  * The documentation for the confusion matrix can be found here: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html
  * The interface and an example for the `ConfusionMatrixDisplay` can be found here: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ConfusionMatrixDisplay.html

In [ ]:
import sklearn.metrics

# get the target class names
classes = trainset_digits.classes

# compute confusion matrix
matrix = sklearn.metrics.confusion_matrix(targets, predictions) # Use predictions and target from the fine-tuned network

# plot confusion matrices
plot_conf_matrix1 = sklearn.metrics.ConfusionMatrixDisplay(matrix, display_labels=classes)
plot_conf_matrix1.plot(xticks_rotation="vertical")
